<a href="https://colab.research.google.com/github/SamTr7/Hackaton/blob/main/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Fine-tuning rápido YOLO para cacao-agricultura (detección con bounding boxes)

Objetivo del hackatón:
- Entrenar un detector que ubique automáticamente zonas de enfermedad en mazorcas de cacao.
- Dataset en carpetas: `Fito`, `Monilia`, `Sana`.
- Etiquetas en formato YOLO por archivo `.txt` junto a cada imagen `.jpg`.

En este notebook construimos el flujo completo:
1. Preparación de datos desde carpetas de clase
2. Conversión a estructura YOLO split (`train/val/test`)
3. Validación de etiquetas
4. Fine-tuning de modelo preentrenado
5. Evaluación, predicción y exportación

In [ ]:
# ======================================
# 0) Descarga del dataset en Google Colab
# ======================================
# Ejecuta esta celda primero en Colab.

# Si da error de módulo no encontrado, descomenta estas líneas:
# !pip install -q dataset-tools
# !pip install -q ultralytics

import dataset_tools as dtools

dtools.download(dataset='Cocoa Diseases', dst_dir='~/dataset-ninja/')
print("Descarga solicitada en ~/dataset-ninja/")

# ======================================
# 0.1) Librerías base del proyecto
# ======================================
import random
import shutil
from pathlib import Path

import torch
import yaml
from ultralytics import YOLO

SEED = 42
random.seed(SEED)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
print("Ultralytics YOLO importado correctamente")

## 1) Configuración del proyecto y rutas

Este notebook asume este origen de datos en Colab:
- `/content/Hackaton_unzipped/Enfermedades Cacao/Fito`
- `/content/Hackaton_unzipped/Enfermedades Cacao/Monilia`
- `/content/Hackaton_unzipped/Enfermedades Cacao/Sana`

Dentro de cada carpeta hay pares `.jpg` + `.txt` con etiquetas YOLO.
El código de abajo arma automáticamente el split `train/val/test` en formato YOLO.

In [ ]:
# ==============================
# Configuración principal
# ==============================
PROJECT_NAME = "cacao_yolo_hackaton"
SEED = 42

# Ruta fuente (tal como indicaste en Colab)
SOURCE_DATASET_DIR = Path("/content/Hackaton_unzipped/Enfermedades Cacao")

# Carpeta de salida en formato YOLO split
YOLO_DATASET_DIR = Path("/content/Hackaton_unzipped/cacao_yolo_dataset")

# Tu dataset actual NO viene pre-spliteado
USE_PRE_SPLIT_DATA = False

# Ajusta este mapping solo si tus .txt usan otro orden de IDs
CLASS_NAMES = ["fito", "monilia", "sana"]
NUM_CLASSES = len(CLASS_NAMES)

RUNS_BASE_DIR = Path("runs")

print("Configuración lista:")
print(f"- SOURCE_DATASET_DIR: {SOURCE_DATASET_DIR}")
print(f"- YOLO_DATASET_DIR: {YOLO_DATASET_DIR}")
print(f"- Clases ({NUM_CLASSES}): {CLASS_NAMES}")

if SOURCE_DATASET_DIR.exists():
    subfolders = sorted([p.name for p in SOURCE_DATASET_DIR.iterdir() if p.is_dir()])
    print(f"- Subcarpetas detectadas: {subfolders}")
else:
    print("[WARN] La ruta fuente aún no existe. Verifica que ya descargaste/descomprimiste el dataset.")

## 2) Preparar dataset YOLO desde carpetas `Fito/Monilia/Sana`

Este bloque toma la estructura original y genera:

```text
/content/Hackaton_unzipped/cacao_yolo_dataset/
  images/
    train/
    val/
    test/
  labels/
    train/
    val/
    test/
```

Notas:
- Cada imagen conserva su `.txt` YOLO asociado.
- Si falta un `.txt`, se crea vacío para no romper el pipeline.
- Se renombra con prefijo de carpeta para evitar colisiones de nombres (ej. `Fito1.jpg` y `Monilia1.jpg`).

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def list_images(folder: Path):
    if not folder.exists():
        return []
    return sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS])

def ensure_dirs(base_dir: Path):
    for split in ["train", "val", "test"]:
        (base_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (base_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

def gather_pairs_from_class_folders(source_dir: Path):
    if not source_dir.exists():
        raise FileNotFoundError(f"No existe la ruta fuente: {source_dir}")

    class_dirs = sorted([d for d in source_dir.iterdir() if d.is_dir()])
    if not class_dirs:
        raise FileNotFoundError(f"No se encontraron subcarpetas de clase en: {source_dir}")

    pairs = []
    for class_dir in class_dirs:
        imgs = list_images(class_dir)
        for img_path in imgs:
            label_path = img_path.with_suffix(".txt")
            pairs.append((img_path, label_path, class_dir.name))

    if not pairs:
        raise FileNotFoundError(f"No se encontraron imágenes válidas en: {source_dir}")

    return pairs, [d.name for d in class_dirs]

def split_items(items, train_ratio=0.7, val_ratio=0.2, test_ratio=0.1, seed=42):
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-9, "Las proporciones deben sumar 1"

    items = items.copy()
    random.seed(seed)
    random.shuffle(items)

    n = len(items)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    return {
        "train": items[:n_train],
        "val": items[n_train:n_train + n_val],
        "test": items[n_train + n_val:],
    }

def make_safe_stem(source_dir: Path, img_path: Path):
    # Usa la ruta relativa para evitar colisiones de nombres entre carpetas de clase.
    rel_no_ext = img_path.relative_to(source_dir).with_suffix("")
    return "_".join(rel_no_ext.parts)

def class_id_distribution(labels_base_dir: Path):
    counts = {}
    for split in ["train", "val", "test"]:
        labels_dir = labels_base_dir / "labels" / split
        if not labels_dir.exists():
            continue
        for txt_path in labels_dir.glob("*.txt"):
            text = txt_path.read_text(encoding="utf-8").strip()
            if not text:
                continue
            for line in text.splitlines():
                parts = line.split()
                if not parts:
                    continue
                try:
                    cls_id = int(float(parts[0]))
                    counts[cls_id] = counts.get(cls_id, 0) + 1
                except ValueError:
                    continue
    return dict(sorted(counts.items(), key=lambda x: x[0]))

def prepare_yolo_from_class_folders(source_dir: Path, out_dir: Path,
                                    train_ratio=0.7, val_ratio=0.2, test_ratio=0.1, seed=42):
    pairs, class_folders = gather_pairs_from_class_folders(source_dir)
    splits = split_items(pairs, train_ratio=train_ratio, val_ratio=val_ratio, test_ratio=test_ratio, seed=seed)

    # Limpieza de salida previa para evitar archivos residuales entre corridas.
    if (out_dir / "images").exists():
        shutil.rmtree(out_dir / "images")
    if (out_dir / "labels").exists():
        shutil.rmtree(out_dir / "labels")

    ensure_dirs(out_dir)

    missing_labels = 0
    for split_name, split_items_list in splits.items():
        for img_path, label_path, _ in split_items_list:
            safe_stem = make_safe_stem(source_dir, img_path)
            dst_img = out_dir / "images" / split_name / f"{safe_stem}{img_path.suffix.lower()}"
            dst_lbl = out_dir / "labels" / split_name / f"{safe_stem}.txt"

            shutil.copy2(img_path, dst_img)

            if label_path.exists():
                shutil.copy2(label_path, dst_lbl)
            else:
                dst_lbl.write_text("", encoding="utf-8")
                missing_labels += 1

    print("Preparación YOLO completada")
    print(f"- Carpeta fuente: {source_dir}")
    print(f"- Carpetas clase detectadas: {class_folders}")
    print(f"- Carpeta salida: {out_dir}")
    print(f"- Total imágenes: {len(pairs)}")
    for split_name, split_items_list in splits.items():
        print(f"  - {split_name}: {len(split_items_list)}")
    print(f"- Labels faltantes rellenados con vacío: {missing_labels}")

    dist = class_id_distribution(out_dir)
    print(f"- Distribución de class_id en labels: {dist}")
    if dist:
        min_id, max_id = min(dist), max(dist)
        if min_id < 0 or max_id >= NUM_CLASSES:
            print(f"[WARN] Hay class_id fuera del rango esperado [0, {NUM_CLASSES - 1}].")

if not USE_PRE_SPLIT_DATA:
    prepare_yolo_from_class_folders(
        source_dir=SOURCE_DATASET_DIR,
        out_dir=YOLO_DATASET_DIR,
        train_ratio=0.7,
        val_ratio=0.2,
        test_ratio=0.1,
        seed=SEED,
    )
else:
    print("Se asume que el dataset ya está en formato YOLO split.")

## 3) Crear archivo dataset.yaml (YOLO)

YOLO necesita un YAML con rutas y nombres de clases.
En esta versión usamos 3 clases:
- `0: fito`
- `1: monilia`
- `2: sana`

Si al inspeccionar tu dataset ves otro mapping de IDs, ajusta `CLASS_NAMES` en la celda 4.

In [ ]:
DATASET_YAML_PATH = YOLO_DATASET_DIR / "dataset.yaml"



dataset_yaml = {

    "path": str(YOLO_DATASET_DIR.resolve()),

    "train": "images/train",

    "val": "images/val",

    "test": "images/test",

    "nc": NUM_CLASSES,

    "names": CLASS_NAMES,

}



with open(DATASET_YAML_PATH, "w", encoding="utf-8") as f:

    yaml.safe_dump(dataset_yaml, f, sort_keys=False, allow_unicode=True)



print(f"dataset.yaml creado en: {DATASET_YAML_PATH.resolve()}")

print(yaml.safe_dump(dataset_yaml, sort_keys=False, allow_unicode=True))


## 4) Validación rápida de anotaciones YOLO



Este paso ayuda a detectar problemas típicos antes de entrenar:

- formato incorrecto en líneas (`5` valores esperados)

- `class_id` fuera de rango

- coordenadas fuera de `[0, 1]`

- cajas con ancho/alto `<= 0`


In [ ]:
def validate_label_file(label_path: Path, num_classes: int):

    errors = []

    text = label_path.read_text(encoding="utf-8").strip()

    if text == "":

        return errors  # archivo vacío es válido para imágenes sanas



    for i, line in enumerate(text.splitlines(), start=1):

        parts = line.split()

        if len(parts) != 5:

            errors.append(f"{label_path.name}: línea {i} -> se esperaban 5 valores, llegaron {len(parts)}")

            continue



        try:

            cls = int(float(parts[0]))

            x, y, w, h = map(float, parts[1:])

        except ValueError:

            errors.append(f"{label_path.name}: línea {i} -> valores no numéricos")

            continue



        if cls < 0 or cls >= num_classes:

            errors.append(f"{label_path.name}: línea {i} -> class_id {cls} fuera de rango [0, {num_classes - 1}]")



        for k, v in zip(["x", "y", "w", "h"], [x, y, w, h]):

            if v < 0 or v > 1:

                errors.append(f"{label_path.name}: línea {i} -> {k}={v:.4f} fuera de [0,1]")



        if w <= 0 or h <= 0:

            errors.append(f"{label_path.name}: línea {i} -> w/h deben ser > 0")



    return errors



def validate_split(split_name: str, root_dir: Path, num_classes: int):

    images_dir = root_dir / "images" / split_name

    labels_dir = root_dir / "labels" / split_name



    if not images_dir.exists() or not labels_dir.exists():

        print(f"[WARN] Split '{split_name}' no encontrado en {root_dir}")

        return



    images = list_images(images_dir)

    label_files = sorted(labels_dir.glob("*.txt"))



    missing_labels = [img for img in images if not (labels_dir / f"{img.stem}.txt").exists()]

    print(f"\nSplit: {split_name}")

    print(f"- Imágenes: {len(images)}")

    print(f"- Labels: {len(label_files)}")

    print(f"- Imágenes sin label .txt: {len(missing_labels)}")



    all_errors = []

    for lf in label_files:

        all_errors.extend(validate_label_file(lf, num_classes))



    if all_errors:

        print(f"- Errores encontrados: {len(all_errors)}")

        print("Primeros 15 errores:")

        for e in all_errors[:15]:

            print("  ", e)

    else:

        print("- Validación OK")



for split in ["train", "val", "test"]:

    validate_split(split, YOLO_DATASET_DIR, NUM_CLASSES)


## 5) Cargar modelo base YOLO (modo rápido 30 min en T4)

Para entrenamiento express con pocas imágenes y GPU T4:
- Modelo recomendado: `yolov8n.pt`
- `imgsz` moderado (`512`) para acelerar
- `time` en entrenamiento para cortar automáticamente alrededor de 30 minutos

Este modo prioriza velocidad y resultado usable de hackatón sobre máxima precisión.

In [ ]:
MODEL_CHECKPOINT = "yolov8n.pt"  # ligero y rápido para T4
model = YOLO(MODEL_CHECKPOINT)

print(f"Modelo base cargado: {MODEL_CHECKPOINT}")
model.info()

# Perfil sprint para pocas imágenes + límite de ~30 minutos
IMG_SIZE = 512
EPOCHS = 200  # tope alto; el tiempo mandará
BATCH = 16 if torch.cuda.is_available() else 8
PATIENCE = 12
DEVICE = 0 if torch.cuda.is_available() else "cpu"
WORKERS = 2
MAX_TRAIN_MINUTES = 28
TRAIN_TIME_HOURS = MAX_TRAIN_MINUTES / 60

print(f"Device: {DEVICE}")
print(f"IMG_SIZE={IMG_SIZE}, BATCH={BATCH}, EPOCHS_MAX={EPOCHS}")
print(f"Límite de tiempo de entrenamiento: {MAX_TRAIN_MINUTES} minutos")

## 6) Fine-tuning del detector (perfil sprint T4)

Este bloque está optimizado para terminar en ~30 minutos:
- `time`: límite de tiempo real del entrenamiento
- `cache=True`: acelera carga de dataset pequeño
- `freeze=10`: acelera ajuste inicial aprovechando pesos preentrenados
- augmentations moderadas para no sobrecargar el tiempo de cómputo

In [ ]:
train_results = model.train(
    data=str(DATASET_YAML_PATH),
    epochs=EPOCHS,
    time=TRAIN_TIME_HOURS,  # corta por tiempo (horas)
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    cache=True,
    amp=True,
    project=str(RUNS_BASE_DIR),
    name=f"{PROJECT_NAME}_fast30m",
    pretrained=True,
    freeze=10,
    optimizer="SGD",
    lr0=5e-3,
    lrf=1e-2,
    weight_decay=5e-4,
    warmup_epochs=1,
    hsv_h=0.010,
    hsv_s=0.4,
    hsv_v=0.2,
    degrees=5.0,
    translate=0.05,
    scale=0.20,
    fliplr=0.5,
    mosaic=0.10,
    mixup=0.0,
    close_mosaic=5,
    patience=PATIENCE,
    seed=SEED,
    deterministic=False,
    val=True,
    plots=False,
)

print("Entrenamiento sprint finalizado")
print(f"Artefactos guardados en: {Path(model.trainer.save_dir).resolve()}")
print(f"Best checkpoint: {Path(model.trainer.best).resolve()}")

## 7) Evaluación del modelo



Cargamos el mejor checkpoint y medimos desempeño sobre validación.

Métricas clave para comparar iteraciones:

- `mAP50`

- `mAP50-95`


In [ ]:
best_model_path = Path(model.trainer.best)

best_model = YOLO(str(best_model_path))



metrics = best_model.val(

    data=str(DATASET_YAML_PATH),

    split="val",

    imgsz=IMG_SIZE,

    conf=0.25,

    iou=0.6,

    device=DEVICE,

    plots=True,

)



print(f"Best checkpoint: {best_model_path}")

print(f"mAP50: {metrics.box.map50:.4f}")

print(f"mAP50-95: {metrics.box.map:.4f}")


## 8) Inferencia visual (detección automática de bbox)



Probamos el mejor modelo en imágenes de `test` para ver si localiza bien las zonas enfermas.


In [ ]:
from IPython.display import Image, display



INFER_SOURCE = YOLO_DATASET_DIR / "images" / "test"

PRED_NAME = f"{PROJECT_NAME}_predict"



pred_results = best_model.predict(

    source=str(INFER_SOURCE),

    imgsz=IMG_SIZE,

    conf=0.25,

    iou=0.6,

    device=DEVICE,

    save=True,

    project=str(RUNS_BASE_DIR),

    name=PRED_NAME,

    exist_ok=True,

    max_det=5,

)



pred_dir = RUNS_BASE_DIR / PRED_NAME

print(f"Predicciones guardadas en: {pred_dir.resolve()}")



# Mostrar algunas predicciones

preview = [p for p in sorted(pred_dir.iterdir()) if p.suffix.lower() in IMG_EXTS][:6]

for p in preview:

    display(Image(filename=str(p)))


## 9) Exportar modelo para despliegue



Exportamos a ONNX para integrarlo en backend, edge devices o pipelines de agricultura digital.


In [ ]:
# Si falla por dependencias, instala: !pip install -q onnx onnxsim

onnx_path = best_model.export(format="onnx", opset=12, simplify=True)

print(f"Modelo exportado en: {onnx_path}")


## 10) Recomendaciones rápidas para mejorar en hackatón



- Etiquetado: revisa cajas ajustadas al borde de la lesión (evita cajas demasiado grandes).

- Balance: mantén proporción razonable entre Fito y Monilia en `train` y `val`.

- Calidad visual: agrega fotos con distintas luces, ángulos y estados de madurez.

- Negativos duros: incluye imágenes sanas parecidas a enfermas para reducir falsos positivos.

- Itera rápido: compara 2-3 corridas cambiando `imgsz`, `epochs`, `conf` y augmentations.



Con este esqueleto ya tienes una base sólida para detección automática de enfermedad en cacao.


In [ ]:
# ==============================
# 11) Top 3 predicciones para diapositivas
# ==============================
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image as PILImage

TOP_K = 3
MIN_CONF = 0.25

# Si no existen predicciones en memoria, las genera en el set de test.
if "pred_results" not in globals() or pred_results is None or len(pred_results) == 0:
    pred_results = best_model.predict(
        source=str(INFER_SOURCE),
        imgsz=IMG_SIZE,
        conf=MIN_CONF,
        iou=0.6,
        device=DEVICE,
        save=False,
        verbose=False,
    )

# Ranking simple por confianza: prioriza max_conf y luego mean_conf.
ranked = []
for r in pred_results:
    if r.boxes is None or len(r.boxes) == 0:
        continue

    confs = r.boxes.conf.detach().cpu().numpy()
    max_conf = float(confs.max())
    mean_conf = float(confs.mean())
    n_boxes = int(len(confs))
    score = 0.7 * max_conf + 0.3 * mean_conf

    ranked.append({
        "result": r,
        "score": score,
        "max_conf": max_conf,
        "mean_conf": mean_conf,
        "n_boxes": n_boxes,
    })

ranked = sorted(
    ranked,
    key=lambda x: (x["score"], x["max_conf"], x["n_boxes"]),
    reverse=True,
)[:TOP_K]

if not ranked:
    print("No se encontraron predicciones con cajas para mostrar.")
else:
    out_dir = RUNS_BASE_DIR / f"{PROJECT_NAME}_best_slides"
    out_dir.mkdir(parents=True, exist_ok=True)

    fig, axes = plt.subplots(1, len(ranked), figsize=(7 * len(ranked), 7))
    if len(ranked) == 1:
        axes = [axes]

    print("Top predicciones seleccionadas para diapositivas:")
    for i, (ax, item) in enumerate(zip(axes, ranked), start=1):
        plotted_bgr = item["result"].plot(line_width=4, font_size=18)
        plotted_rgb = plotted_bgr[:, :, ::-1]

        src_name = Path(item["result"].path).name
        out_file = out_dir / f"top{i}_{Path(item['result'].path).stem}.jpg"
        PILImage.fromarray(plotted_rgb).save(out_file, quality=95)

        ax.imshow(plotted_rgb)
        ax.axis("off")
        ax.set_title(
            f"Top {i} | score={item['score']:.3f}\n"
            f"max={item['max_conf']:.3f} | mean={item['mean_conf']:.3f} | boxes={item['n_boxes']}\n"
            f"{src_name}",
            fontsize=11,
        )

        print(
            f"{i}. {src_name} | score={item['score']:.3f} | "
            f"max={item['max_conf']:.3f} | guardada en: {out_file}"
        )

    plt.tight_layout()
    collage_path = out_dir / "top3_collage.png"
    fig.savefig(collage_path, dpi=220, bbox_inches="tight")
    plt.show()

    print(f"Collage para diapositiva: {collage_path}")